# Overview of anomalies

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from anomaly.utils import AnomalyOverlapAnalyzer
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Constants

In [3]:
mse_cols = ['mse', 'mse_filter_250', 'mse_97', 'mse_filter_250_97']
mse_rel_cols = ['mse_rel', 'mse_filter_250_rel', 'mse_97_rel', 'mse_filter_250_97_rel']

In [4]:
mse_family = ['mse', 'mse_97', 'mse_filter_250', 'mse_filter_250_97']
chi_sq_family = ['mse_rel', 'mse_97_rel', 'mse_filter_250_rel', 'mse_filter_250_97_rel']

# Custom functions

## IDs top anomalies

In [5]:
def get_ids(score, df, quantile=99, n_top=None, use_ntop=False):

    if use_ntop is False:
        
        quantile *= 0.01
        thresh = df[score].quantile(quantile)
        ids = set(df[df[score] > thresh].index)
        
    else:

        ids = set(
            df[score].sort_values(
                ascending=False
            ).iloc[:n_top].index
        )

    return ids

In [6]:
def top_unique_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    for score in scores_list:

        ids_top_dict[score] = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

    n_top = len(ids_top_dict[score])

    unique_ids_dict = AnomalyOverlapAnalyzer.get_unique_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    for score in scores_list:
        n_unique = len(unique_ids_dict[score])

        unique_pct = n_unique/n_top*100
        
        print(f"Unique to {score}:\n{n_unique} --> {unique_pct:.2f}%")

    return unique_ids_dict, ids_top_dict


In [7]:
def top_common_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    for score in scores_list:

        ids_top_dict[score] = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

    n_top = len(ids_top_dict[score])

    common_ids_dict = AnomalyOverlapAnalyzer.get_core_common_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    n_common = len(common_ids_dict)
    common_pct = n_common/n_top*100 
    print(f"N common:\n{n_common} -- > {common_pct}%")

    return common_ids_dict, ids_top_dict

## Figures

In [8]:
def anomaly_plot(wave, specs, objids, ranks, save_to):

    fig, ax = plt.subplots(
        figsize=(10, 5)
    )

    for spec, objid, rank in zip(specs, objids, ranks):

        print(f'Rank {rank:03d}', end='\r')

        ax.clear()

        ax.plot(wave, spec, color="black", label=f'Rank: {rank}')

        ax.minorticks_on()
        ax.set_xlabel(r"$\lambda$ [nm]")
        ax.set_title(f"Object ID: {objid}")

        ax.legend(
            loc='upper left',
            frameon=False,
        )

        fig.savefig(
            f"{save_to}/{rank:03d}_{objid}.jpeg",
            bbox_inches='tight'
        )

    plt.close(fig)

# Config

## Directories

In [9]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
bin_ids = [f"bin_{i:02d}" for i in range(4)]
#
ch_4_dir = f"{thesis_dir}/chapters/04_figures"

## Data

In [10]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

In [11]:
bin_id = 'bin_03'
score_03_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    index_col='specobjid'
)
n_spec = score_03_df.shape[0]
n_top_1_pct = int(n_spec*0.01)
n_top_1_pct, n_spec

(1818, 181850)

## Append rank per score

In [12]:
rank = np.arange(score_03_df.shape[0])
score_03_rank_df = score_03_df.copy()
# score_rank_df
for col in score_03_df.columns:

    index_sorted = score_03_df.sort_values(
        by=col, ascending=False
    ).index

    score_03_rank_df.loc[index_sorted, f'rank_{col}'] = rank
    score_03_rank_df[f'rank_{col}'].astype(int)

In [13]:
score = 'mse_97'
score_03_rank_df[[score, f'rank_{score}']].sort_values(
    by=score, ascending=False
).head(10)

,mse_97,rank_mse_97
specobjid,,
1919783100783552512,14.380249,0.0
2245159102839810048,13.726358,1.0
2930814289679771648,11.061983,2.0
1071977423131666432,11.050510,3.0
969534273977608192,9.661587,4.0
2008626123889469440,9.116298,5.0
1530151624956209152,8.005217,6.0
2811439757043722240,8.000128,7.0
1935588031146780672,7.497642,8.0


## Model

In [14]:
ae_03 = AutoEncoder(
    reload=True,
    reload_from=f"{models_dir}/bin_03/winner",
)

In [15]:
ae_03.get_architecture_and_model_str()

['256_128_64_12_64_128_256', 'infoVae_rec_3776_alpha_0_lambda_9']

# Figures top 1000

```python
n_top = 1000
all_scores = mse_cols + mse_rel_cols
plt.ioff()

for score in all_scores:

    specids_top_1 = score_03_df[score].sort_values(
        ascending=False
    ).index.to_numpy()[:n_top]

    ranks = np.zeros(n_top).astype(int)

    specs_top_1 = np.empty((n_top, wave.size))

    for i, objid in enumerate(specids_top_1):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs_top_1[i, :] = spectra[spec_idx, :]

        ranks[i] = i

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/bin_03/figs/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs_top_1,
        objids=specids_top_1, ranks=ranks,
        save_to=save_to
    )
```

# No free lunch theorem

## IDs per score

In [16]:
ids_top_dict = {}

all_scores = mse_cols + mse_rel_cols

for score in all_scores:

    ids_top_dict[score] = get_ids(
        score=score,
        df=score_03_df.copy(),
        quantile=99,
        n_top=1000,
        use_ntop=False
    )

n_top_1 = len(ids_top_dict[score])
n_top_1

1819

# Distinct IDs

## MSE family

In [17]:
mse_unique_ids_dict, mse_ids_top_dict = top_unique_ids(
    scores_df=score_03_df.copy(),
    scores_list=mse_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

Unique to mse:
84 --> 4.62%
Unique to mse_filter_250:
68 --> 3.74%
Unique to mse_97:
11 --> 0.60%
Unique to mse_filter_250_97:
15 --> 0.82%


In [48]:
# score = 'mse'
# unique_mse_ids = list(unique_mse_ids_dict[score])
# score_03_rank_df.loc[
#     unique_mse_ids, [score, f'rank_{score}']
# ].sort_values(by=score, ascending=False)

## Chi family

In [18]:
chi_unique_ids_dict, chi_ids_top_dict = top_unique_ids(
    scores_df=score_03_df.copy(),
    scores_list=mse_rel_cols,
    quantile=90,
    # n_top=1000, use_ntop=True
)

Unique to mse_rel:
328 --> 1.80%
Unique to mse_filter_250_rel:
196 --> 1.08%
Unique to mse_97_rel:
85 --> 0.47%
Unique to mse_filter_250_97_rel:
114 --> 0.63%


In [50]:
# score = 'mse_97_rel'
# unique_mse_ids = list(unique_chi_ids_dict[score])
# score_03_rank_df.loc[
#     unique_mse_ids, [score, f'rank_{score}']
# ].sort_values(by=score, ascending=False)

## All scores

In [19]:
all_scores = mse_cols + mse_rel_cols

chi_unique_ids_dict, chi_ids_top_dict = top_unique_ids(
    scores_df=score_03_df.copy(),
    scores_list=all_scores,
    quantile=90,
    # n_top=1000, use_ntop=True
)

Unique to mse:
211 --> 1.16%
Unique to mse_filter_250:
77 --> 0.42%
Unique to mse_97:
34 --> 0.19%
Unique to mse_filter_250_97:
26 --> 0.14%
Unique to mse_rel:
167 --> 0.92%
Unique to mse_filter_250_rel:
119 --> 0.65%
Unique to mse_97_rel:
63 --> 0.35%
Unique to mse_filter_250_97_rel:
60 --> 0.33%


## Figures

In [28]:
all_scores = mse_cols + mse_rel_cols
plt.ioff()

for score in all_scores:

    specids = list(unique_all_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_03_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/bin_03/figs/unique_all/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

# Common IDs

## MSE family

In [20]:
mse_common_ids_dict, mse_ids_top_dict = top_common_ids(
    scores_df=score_03_df.copy(), scores_list=mse_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
868 -- > 47.71852666300165%


## Chi Family

In [21]:
chi_common_ids_dict, chi_ids_top_dict = top_common_ids(
    scores_df=score_03_df.copy(), scores_list=mse_rel_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
921 -- > 50.63221550302364%


## All scores

In [31]:
all_scores = mse_cols + mse_rel_cols 
all_common_ids_dict, all_ids_top_dict = top_common_ids(
    scores_df=score_03_df.copy(), scores_list=all_scores,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
557 -- > 30.621220450797143%


### Figures

In [35]:
all_scores = mse_cols + mse_rel_cols
plt.ioff()


specids = list(common_all_ids)
specids = np.array(specids, dtype=int)

ranks = score_03_rank_df.loc[
    specids, 'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/bin_03/figs/common_all"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)